# Текущий контроль 7–8 (реализация): логистическая регрессия для оценки страхового риска

**Задание.** Реализовать логистическую регрессию двумя способами: (а) максимизация логарифмического правдоподобия градиентным **подъёмом** и (б) минимизация кросс-энтропии градиентным **спуском**. Сравнить время, accuracy и F1 при одинаковых начальных параметрах, числе шагов и скорости обучения. Применить оба способа к `insclass_train.csv` и подготовить прогноз `target` для `insclass_test.csv`.

В проекте сейчас есть только обучающий CSV. Сравнение проводится на отложенных 20% известных договоров. Последняя ячейка создаст `insclass_test_predictions.csv`, когда `insclass_test.csv` будет помещён рядом с ноутбуком.

## 1. Модель и две эквивалентные функции цели

Каждому договору соответствует вектор признаков $\mathbf{x}_i$, а $y_i\in\{0,1\}$: 1 означает рисковый договор. Логистическая регрессия оценивает вероятность класса 1:

$$z_i=\mathbf{x}_i^T\boldsymbol\beta+b,\qquad p_i=\sigma(z_i)=\frac{1}{1+e^{-z_i}}.$$

Чтобы редкий класс влиял на обучение, используем веса $q_i=n/(2n_{y_i})$, где $n_{y_i}$ — число обучающих договоров соответствующего класса. Сумма весов $Q=\sum_iq_i$. Добавляем одинаковую L2-регуляризацию к обоим методам:

$$J_{CE}(\boldsymbol\beta,b)=\frac{1}{Q}\sum_iq_i\left[\log(1+e^{z_i})-y_iz_i\right]+\frac{\lambda}{2}\lVert\boldsymbol\beta\rVert_2^2,$$

$$J_{LL}(\boldsymbol\beta,b)=-J_{CE}(\boldsymbol\beta,b).$$

Тогда градиенты тоже противоположны: $\nabla J_{LL}=-\nabla J_{CE}$. При одинаковых начальных коэффициентах, шаге $\eta$ и числе итераций подъём по $J_{LL}$ и спуск по $J_{CE}$ должны привести к одинаковому решению. Это математическая проверка реализации. Для устойчивых вычислений используются `expit` и `logaddexp`.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import expit
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Загрузка и анализ данных

Файл содержит 151 406 договоров и 28 исходных признаков. Некоторые столбцы категориальные, многие числовые имеют пропуски. `variable_3` постоянна, а `variable_15` заполнена лишь для 13 договоров; эти два поля исключаем. Остальные договоры сохраняем: одинаковые признаки могут соответствовать разным реальным договорам.

In [ ]:
train_file = Path('insclass_train.csv')
if not train_file.is_file():
    raise FileNotFoundError('Поместите insclass_train.csv рядом с ноутбуком.')

insurance = pd.read_csv(train_file)
if 'target' not in insurance.columns:
    raise ValueError('В обучающем CSV отсутствует target.')
if insurance['target'].isna().any() or set(insurance['target'].unique()) != {0, 1}:
    raise ValueError('target должен содержать только 0 и 1 без пропусков.')

drop_columns = ['variable_3', 'variable_15']
X = insurance.drop(columns=['target', *drop_columns])
y = insurance['target'].astype(int)

print(f'Договоров: {len(insurance)}; признаков после отбора: {X.shape[1]}')
print('Распределение целевого класса:')
display(y.value_counts().sort_index().rename('Количество').to_frame())
print(f'Доля рисковых договоров: {y.mean():.2%}')
print('Наиболее часто пропущенные признаки:')
display(X.isna().sum().sort_values(ascending=False).head(8).rename('Пропусков').to_frame())
print(f'Заполненных значений variable_15: {insurance["variable_15"].notna().sum()}')

## 3. Отложенная проверка и подготовка признаков

Отложим 20% договоров с сохранением доли классов. Подготовку признаков обучаем **только на 80% train**, иначе медианы, масштабы и словари категорий увидят тестовую часть.

- Числовые признаки: пропуски заменяем медианой, добавляем индикаторы пропусков и стандартизируем $x'_j=(x_j-\mu_j)/s_j$.
- Категории, включая идентификатор года `variable_4`: пропуски заменяем самой частой категорией и кодируем one-hot. Неизвестные категории на проверке или в будущем тестовом CSV игнорируются.
- Результат храним в разреженной матрице: её размерность велика из-за марок и моделей автомобилей, но в каждой строке заполнена только малая часть столбцов.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Идентификаторы кодируем как категории, чтобы не приписывать им числовой порядок.
categorical_columns = [
    'variable_1', 'variable_4', 'variable_5', 'variable_20',
    'variable_21', 'variable_22', 'variable_28',
]
numeric_columns = [col for col in X.columns if col not in categorical_columns]

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', make_pipeline(
            SimpleImputer(strategy='median', add_indicator=True),
            StandardScaler(),
        ), numeric_columns),
        ('categorical', make_pipeline(
            SimpleImputer(strategy='most_frequent'),
            OneHotEncoder(handle_unknown='ignore'),
        ), categorical_columns),
    ],
    sparse_threshold=1.0,
)

# fit_transform изучает только train; transform применяет его правила к valid.
X_train_ready = preprocessor.fit_transform(X_train).tocsr()
X_valid_ready = preprocessor.transform(X_valid).tocsr()
y_train_array = y_train.to_numpy(dtype=float)
y_valid_array = y_valid.to_numpy(dtype=int)

print(f'Train: {X_train_ready.shape}; validation: {X_valid_ready.shape}')
print(f'Ненулевых элементов в train: {X_train_ready.nnz:,}')
print(f'Доля положительного класса: train {y_train.mean():.2%}, '
      f'validation {y_valid.mean():.2%}')

## 4. Собственная реализация градиентного обучения

Обозначим $r_i=q_i(p_i-y_i)/Q$. Градиент кросс-энтропии равен

$$\nabla_{\boldsymbol\beta}J_{CE}=X^T\mathbf r+\lambda\boldsymbol\beta,\qquad
\frac{\partial J_{CE}}{\partial b}=\sum_i r_i.$$

Для спуска вычитаем $\eta\nabla J_{CE}$. Для подъёма добавляем $\eta\nabla J_{LL}$, где $\nabla J_{LL}=-\nabla J_{CE}$. Свободный коэффициент $b$ не штрафуется. Класс реализует оба шага явно, без готового `LogisticRegression`.

In [ ]:
class LogisticRegressionGradient:
    # Логистическая регрессия: подъём LL или спуск CE с полным градиентом.

    def __init__(self, learning_rate=0.3, n_steps=250, l2=0.001, method='CE'):
        if method not in {'CE', 'LL'}:
            raise ValueError("method должен быть 'CE' или 'LL'.")
        self.learning_rate = learning_rate
        self.n_steps = n_steps
        self.l2 = l2
        self.method = method

    def fit(self, features, target, sample_weight):
        target = np.asarray(target, dtype=float)
        sample_weight = np.asarray(sample_weight, dtype=float)
        if features.shape[0] != len(target) or len(target) != len(sample_weight):
            raise ValueError('Размеры X, y и весов должны совпадать.')

        self.coef_ = np.zeros(features.shape[1], dtype=float)
        self.intercept_ = 0.0
        self.objective_history_ = []
        normalizer = sample_weight.sum()

        for _ in range(self.n_steps):
            z = features @ self.coef_ + self.intercept_
            probability = expit(z)

            # logaddexp(0, z) - y*z = -log P(y|x), устойчиво при больших |z|.
            ce = np.dot(sample_weight, np.logaddexp(0, z) - target * z) / normalizer
            ce += 0.5 * self.l2 * np.dot(self.coef_, self.coef_)
            self.objective_history_.append(ce if self.method == 'CE' else -ce)

            if self.method == 'CE':
                # Градиентный спуск: минимизируем кросс-энтропию.
                residual = sample_weight * (probability - target) / normalizer
                gradient_coef = features.T @ residual + self.l2 * self.coef_
                gradient_intercept = residual.sum()
                self.coef_ -= self.learning_rate * gradient_coef
                self.intercept_ -= self.learning_rate * gradient_intercept
            else:
                # Градиентный подъём: максимизируем log-likelihood.
                residual = sample_weight * (target - probability) / normalizer
                gradient_coef = features.T @ residual - self.l2 * self.coef_
                gradient_intercept = residual.sum()
                self.coef_ += self.learning_rate * gradient_coef
                self.intercept_ += self.learning_rate * gradient_intercept

        return self

    def predict_proba_positive(self, features):
        return expit(features @ self.coef_ + self.intercept_)

    def predict(self, features, threshold=0.5):
        return (self.predict_proba_positive(features) >= threshold).astype(int)

## 5. Одинаковые условия для обоих методов

Так как класс 1 составляет менее 10% данных, обычная accuracy может быть высокой даже у модели, которая всегда отвечает 0. Веса $q_c=n/(2n_c)$ делают суммарный вес обоих классов одинаковым и дают модели стимул находить рисковые договоры. У обоих методов **одни и те же** веса, старт из нулей, $\eta=0{,}3$, $\lambda=0{,}001$ и 250 шагов. Время измеряется только для цикла обучения, без загрузки и подготовки данных.

In [ ]:
class_counts = np.bincount(y_train_array.astype(int), minlength=2)
class_weights = len(y_train_array) / (2 * class_counts)
sample_weights = class_weights[y_train_array.astype(int)]
print(f'Веса классов: 0 → {class_weights[0]:.3f}, 1 → {class_weights[1]:.3f}')

settings = dict(learning_rate=0.3, n_steps=250, l2=0.001)
models = {}
elapsed = {}

for method in ('LL', 'CE'):
    model = LogisticRegressionGradient(method=method, **settings)
    start = perf_counter()
    model.fit(X_train_ready, y_train_array, sample_weights)
    elapsed[method] = perf_counter() - start
    models[method] = model

print(f'Время подъёма LL: {elapsed["LL"]:.3f} с')
print(f'Время спуска CE: {elapsed["CE"]:.3f} с')
print('Максимальная разница коэффициентов:',
      np.max(np.abs(models['LL'].coef_ - models['CE'].coef_)))
print('Разница свободных коэффициентов:',
      abs(models['LL'].intercept_ - models['CE'].intercept_))

## 6. Сравнение на отложенных договорах

Считаем `accuracy=(TP+TN)/(TP+TN+FP+FN)` и $F1=2PR/(P+R)$ для **рискового класса 1**, где $P=TP/(TP+FP)$ и $R=TP/(TP+FN)$. Порог вероятности фиксирован на 0,5. Дополнительно показываем ROC-AUC и простое правило «всегда 0». Метрики обоих градиентных методов должны совпасть; небольшая разница во времени вызвана обычной вариацией замера.

In [ ]:
comparison_rows = []
valid_predictions = {}

for method, model in models.items():
    probability = model.predict_proba_positive(X_valid_ready)
    prediction = model.predict(X_valid_ready)
    valid_predictions[method] = prediction
    comparison_rows.append({
        'Метод': 'Подъём LL' if method == 'LL' else 'Спуск CE',
        'Время, с': elapsed[method],
        'Accuracy': accuracy_score(y_valid_array, prediction),
        'F1 класса 1': f1_score(y_valid_array, prediction, zero_division=0),
        'Precision класса 1': precision_score(y_valid_array, prediction, zero_division=0),
        'Recall класса 1': recall_score(y_valid_array, prediction, zero_division=0),
        'ROC-AUC': roc_auc_score(y_valid_array, probability),
    })

baseline = DummyClassifier(strategy='most_frequent').fit(X_train_ready, y_train_array)
baseline_prediction = baseline.predict(X_valid_ready)
comparison_rows.append({
    'Метод': 'Всегда класс 0',
    'Время, с': np.nan,
    'Accuracy': accuracy_score(y_valid_array, baseline_prediction),
    'F1 класса 1': f1_score(y_valid_array, baseline_prediction, zero_division=0),
    'Precision класса 1': precision_score(y_valid_array, baseline_prediction, zero_division=0),
    'Recall класса 1': recall_score(y_valid_array, baseline_prediction, zero_division=0),
    'ROC-AUC': 0.5,
})
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(3))

assert np.array_equal(valid_predictions['LL'], valid_predictions['CE'])
assert np.allclose(models['LL'].coef_, models['CE'].coef_, atol=1e-12)

cm = confusion_matrix(y_valid_array, valid_predictions['CE'], labels=[0, 1])
print('Матрица ошибок CE (строки — фактический класс, столбцы — прогноз):')
display(pd.DataFrame(cm, index=['Факт 0', 'Факт 1'],
                     columns=['Прогноз 0', 'Прогноз 1']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# LL максимизируется, поэтому для общего графика берём -LL: это та же CE.
axes[0].plot(-np.asarray(models['LL'].objective_history_), label='−LL: подъём')
axes[0].plot(models['CE'].objective_history_, linestyle='--', label='CE: спуск')
axes[0].set(xlabel='Шаг', ylabel='Кросс-энтропия с L2',
            title='Ход оптимизации')
axes[0].legend()

ConfusionMatrixDisplay(cm, display_labels=['0: нерисковый', '1: рисковый']).plot(
    ax=axes[1], cmap='Blues', colorbar=False, values_format='d'
)
axes[1].set_title('Матрица ошибок на validation')
fig.tight_layout()
plt.show()

## 7. Прогноз для отдельного тестового файла

Если `insclass_test.csv` появится рядом с ноутбуком, следующая ячейка обучит подготовку признаков и итоговую CE-модель на **всех 151 406 размеченных договорах**, затем сохранит один столбец `target` в `insclass_test_predictions.csv` в исходном порядке строк. Метод LL даёт те же коэффициенты при одинаковых настройках; для итогового прогноза достаточно одного из них. Пока тестового файла нет, ячейка только сообщает об этом.

In [ ]:
test_file = Path('insclass_test.csv')
if test_file.is_file():
    test_contracts = pd.read_csv(test_file)
    missing_columns = sorted(set(X.columns) - set(test_contracts.columns))
    if missing_columns:
        raise ValueError(f'В тестовом файле отсутствуют признаки: {missing_columns}')

    # Порядок и состав столбцов совпадают с обучением; лишние поля не используем.
    test_features = test_contracts.loc[:, X.columns]
    final_preprocessor = clone(preprocessor)
    X_all_ready = final_preprocessor.fit_transform(X).tocsr()
    X_test_ready = final_preprocessor.transform(test_features).tocsr()

    y_all = y.to_numpy(dtype=float)
    counts_all = np.bincount(y_all.astype(int), minlength=2)
    weights_all = len(y_all) / (2 * counts_all)
    sample_weights_all = weights_all[y_all.astype(int)]

    final_model = LogisticRegressionGradient(method='CE', **settings)
    final_model.fit(X_all_ready, y_all, sample_weights_all)
    test_prediction = final_model.predict(X_test_ready)

    output_file = Path('insclass_test_predictions.csv')
    pd.DataFrame({'target': test_prediction}).to_csv(output_file, index=False)
    print(f'Сохранено {len(test_prediction)} прогнозов в {output_file.resolve()}')
    print(f'Среди них рисковых договоров: {test_prediction.sum()}')
else:
    print('insclass_test.csv пока отсутствует; прогноз для него будет создан после добавления файла.')

## 8. Выводы

Подъём по логарифмическому правдоподобию и спуск по кросс-энтропии реализуют **одни и те же обновления с противоположными знаками**. На одинаковом разбиении, с одинаковыми весами и настройками они дают практически совпадающие коэффициенты и одинаковые прогнозы. Время выполнения следует сравнивать по выводу ячейки: оно зависит от машины и текущей нагрузки.

Положительный класс составляет около 9,5% договоров. Поэтому правило «всегда 0» имеет accuracy **0,904**, но F1 рискового класса равен нулю. Взвешенная логистическая регрессия имеет accuracy **0,695** и F1 класса 1 **0,245**: она нашла **1499 из 2892** рисковых договоров (recall **0,518**), но также ошибочно пометила рисковыми **7843** нерисковых. Это явный компромисс между поиском редкого класса и ложными тревогами. Матрица ошибок показывает обе стороны результата. Численные результаты и ограничения модели приведены также в README.